# 0. Download the dataset

In [4]:
import os, urllib.request

os.makedirs("data", exist_ok=True)
base = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
for f in ["KDDTrain+.txt", "KDDTest+.txt"]:
    path = os.path.join("data", f)
    if not os.path.exists(path):
        urllib.request.urlretrieve(base + f, path)

print("Working folder:", os.getcwd())
print("Files:", os.listdir("data"))

Working folder: C:\Users\20050\Jupyter\NIDS
Files: ['.ipynb_checkpoints', 'KDDTest+.txt', 'KDDTrain+.txt']


# 1. Import Libraries

In [1]:
import json, time, warnings
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, classification_report)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
SEED = 42

# 2. Column Names


In [5]:
cols = ("duration protocol_type service flag src_bytes dst_bytes land wrong_fragment urgent hot num_failed_logins "
        "logged_in num_compromised root_shell su_attempted num_root num_file_creations num_shells num_access_files "
        "num_outbound_cmds is_host_login is_guest_login count srv_count serror_rate srv_serror_rate rerror_rate "
        "srv_rerror_rate same_srv_rate diff_srv_rate srv_diff_host_rate dst_host_count dst_host_srv_count "
        "dst_host_same_srv_rate dst_host_diff_srv_rate dst_host_same_src_port_rate dst_host_srv_diff_host_rate "
        "dst_host_serror_rate dst_host_srv_serror_rate dst_host_rerror_rate dst_host_srv_rerror_rate").split() + ["label", "difficulty"]
len(cols)


43

In [6]:
cols

['duration',
 'protocol_type',
 'service',
 'flag',
 'src_bytes',
 'dst_bytes',
 'land',
 'wrong_fragment',
 'urgent',
 'hot',
 'num_failed_logins',
 'logged_in',
 'num_compromised',
 'root_shell',
 'su_attempted',
 'num_root',
 'num_file_creations',
 'num_shells',
 'num_access_files',
 'num_outbound_cmds',
 'is_host_login',
 'is_guest_login',
 'count',
 'srv_count',
 'serror_rate',
 'srv_serror_rate',
 'rerror_rate',
 'srv_rerror_rate',
 'same_srv_rate',
 'diff_srv_rate',
 'srv_diff_host_rate',
 'dst_host_count',
 'dst_host_srv_count',
 'dst_host_same_srv_rate',
 'dst_host_diff_srv_rate',
 'dst_host_same_src_port_rate',
 'dst_host_srv_diff_host_rate',
 'dst_host_serror_rate',
 'dst_host_srv_serror_rate',
 'dst_host_rerror_rate',
 'dst_host_srv_rerror_rate',
 'label',
 'difficulty']

# 3. Attack name to attack family

In [7]:
FAMILY = {}
for fam, names in {
    "DoS": "back land neptune pod smurf teardrop apache2 udpstorm processtable worm mailbomb",
    "Probe": "satan ipsweep nmap portsweep mscan saint",
    "R2L": "guess_passwd ftp_write imap phf multihop warezmaster warezclient spy xlock xsnoop snmpguess snmpgetattack httptunnel sendmail named",
    "U2R": "buffer_overflow loadmodule rootkit perl sqlattack xterm ps"}.items():
    for n in names.split(): FAMILY[n] = fam
FAMILY["normal"] = "Normal"

In [8]:
pd.Series(FAMILY).head(10)

back            DoS
land            DoS
neptune         DoS
pod             DoS
smurf           DoS
teardrop        DoS
apache2         DoS
udpstorm        DoS
processtable    DoS
worm            DoS
dtype: object

In [9]:
pd.Series(FAMILY).value_counts()

R2L       15
DoS       11
U2R        7
Probe      6
Normal     1
Name: count, dtype: int64

# 4. Load the data

In [11]:
pd.read_csv("data/KDDTrain+.txt", header=None).head(5)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0.0,0.0,0.0,0.0,1.00,0.00,0.00,150,25,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,1,0.0,0.0,0.0,0.0,0.08,0.15,0.00,255,1,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,123,6,1.0,1.0,0.0,0.0,0.05,0.07,0.00,255,26,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,5,5,0.2,0.2,0.0,0.0,1.00,0.00,0.00,30,255,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,30,32,0.0,0.0,0.0,0.0,1.00,0.00,0.09,255,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [12]:
tr = pd.read_csv("data/KDDTrain+.txt", header=None, names=cols)
te = pd.read_csv("data/KDDTest+.txt", header=None, names=cols)
tr.head(5)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0.0,0.0,0.0,0.0,1.00,0.00,0.00,150,25,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,1,0.0,0.0,0.0,0.0,0.08,0.15,0.00,255,1,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,123,6,1.0,1.0,0.0,0.0,0.05,0.07,0.00,255,26,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,5,5,0.2,0.2,0.0,0.0,1.00,0.00,0.00,30,255,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,30,32,0.0,0.0,0.0,0.0,1.00,0.00,0.09,255,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [13]:
te.head(5)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,229,10,0.0,0.00,1.0,1.0,0.04,0.06,0.00,255,10,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21
1,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,136,1,0.0,0.00,1.0,1.0,0.01,0.06,0.00,255,1,0.00,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.00,0.0,0.0,1.00,0.00,0.00,134,86,0.61,0.04,0.61,0.02,0.0,0.0,0.00,0.00,normal,21
3,0,icmp,eco_i,SF,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,65,0.0,0.00,0.0,0.0,1.00,0.00,1.00,3,57,1.00,0.00,1.00,0.28,0.0,0.0,0.00,0.00,saint,15
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,8,0.0,0.12,1.0,0.5,1.00,0.00,0.75,29,86,0.31,0.17,0.03,0.02,0.0,0.0,0.83,0.71,mscan,11


In [14]:
tr.shape, te.shape

((125973, 43), (22544, 43))

# 5. Add the answers to columns

In [15]:
for d in (tr, te):
    d["family"] = d.label.map(FAMILY); d["attack"] = (d.label != "normal").astype(int)
assert tr.family.notna().all() and te.family.notna().all()
tr[["protocol_type", "service", "src_bytes", "label", "family", "attack"]].head(5)

,protocol_type,service,src_bytes,label,family,attack
0,tcp,ftp_data,491,normal,Normal,0
1,udp,other,146,normal,Normal,0
2,tcp,private,0,neptune,DoS,1
3,tcp,http,232,normal,Normal,0
4,tcp,http,199,normal,Normal,0


In [16]:
tr.family.value_counts()

family
Normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64

In [17]:
te.family.value_counts()


family
Normal    9711
DoS       7460
R2L       2885
Probe     2421
U2R         67
Name: count, dtype: int64

# 6. Separate features from answers

In [18]:
feats = cols[:41]
cat = ["protocol_type", "service", "flag"]; num = [c for c in feats if c not in cat]
Xtr, Xte = tr[feats], te[feats]
Xtr.head(5)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0.0,0.0,0.0,0.0,1.00,0.00,0.00,150,25,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00
1,0,udp,other,SF,146,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,1,0.0,0.0,0.0,0.0,0.08,0.15,0.00,255,1,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00
2,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,123,6,1.0,1.0,0.0,0.0,0.05,0.07,0.00,255,26,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00
3,0,tcp,http,SF,232,8153,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,5,5,0.2,0.2,0.0,0.0,1.00,0.00,0.00,30,255,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01
4,0,tcp,http,SF,199,420,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,30,32,0.0,0.0,0.0,0.0,1.00,0.00,0.09,255,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [19]:
cat, len(num)

(['protocol_type', 'service', 'flag'], 38)

# 7. Summary facts and unseen attack types

In [20]:
R = {"train_rows": len(tr), "test_rows": len(te), "n_features": 41,
     "train_attack_share": float(tr.attack.mean()), "test_attack_share": float(te.attack.mean()),
     "train_family_counts": tr.family.value_counts().to_dict(), "test_family_counts": te.family.value_counts().to_dict(),
     "test_attack_types_unseen_in_training": sorted(set(te.label) - set(tr.label)),
     "test_rows_with_unseen_attack_types": int((~te.label.isin(set(tr.label))).sum())}
print(json.dumps(R, indent=1))

{
 "train_rows": 125973,
 "test_rows": 22544,
 "n_features": 41,
 "train_attack_share": 0.4654171925730117,
 "test_attack_share": 0.5692423704755145,
 "train_family_counts": {
  "Normal": 67343,
  "DoS": 45927,
  "Probe": 11656,
  "R2L": 995,
  "U2R": 52
 },
 "test_family_counts": {
  "Normal": 9711,
  "DoS": 7460,
  "R2L": 2885,
  "Probe": 2421,
  "U2R": 67
 },
 "test_attack_types_unseen_in_training": [
  "apache2",
  "httptunnel",
  "mailbomb",
  "mscan",
  "named",
  "processtable",
  "ps",
  "saint",
  "sendmail",
  "snmpgetattack",
  "snmpguess",
  "sqlattack",
  "udpstorm",
  "worm",
  "xlock",
  "xsnoop",
  "xterm"
 ],
 "test_rows_with_unseen_attack_types": 3750
}


In [21]:
te[te.label.isin(R["test_attack_types_unseen_in_training"])].label.value_counts()

label
mscan            996
apache2          737
processtable     685
snmpguess        331
saint            319
mailbomb         293
snmpgetattack    178
httptunnel       133
named             17
ps                15
sendmail          14
xterm             13
xlock              9
xsnoop             4
worm               2
sqlattack          2
udpstorm           2
Name: count, dtype: int64

# 8. Preprocessing (scaling and one-hot encoding)

In [22]:
def prep():
    return ColumnTransformer([("num", StandardScaler(), num),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])

In [23]:
demo = prep()
clean = demo.fit_transform(Xtr)
clean = clean.toarray() if hasattr(clean, "toarray") else clean
clean_df = pd.DataFrame(clean, columns=demo.get_feature_names_out())
Xtr.shape, clean_df.shape

((125973, 41), (125973, 122))

In [24]:
clean_df.head(5)

,num__duration,num__src_bytes,num__dst_bytes,num__land,num__wrong_fragment,num__urgent,num__hot,num__num_failed_logins,num__logged_in,num__num_compromised,num__root_shell,num__su_attempted,num__num_root,num__num_file_creations,num__num_shells,num__num_access_files,num__num_outbound_cmds,num__is_host_login,num__is_guest_login,num__count,num__srv_count,num__serror_rate,num__srv_serror_rate,num__rerror_rate,num__srv_rerror_rate,num__same_srv_rate,num__diff_srv_rate,num__srv_diff_host_rate,num__dst_host_count,num__dst_host_srv_count,num__dst_host_same_srv_rate,num__dst_host_diff_srv_rate,num__dst_host_same_src_port_rate,num__dst_host_srv_diff_host_rate,num__dst_host_serror_rate,num__dst_host_srv_serror_rate,num__dst_host_rerror_rate,num__dst_host_srv_rerror_rate,cat__protocol_type_icmp,cat__protocol_type_tcp,cat__protocol_type_udp,cat__service_IRC,cat__service_X11,cat__service_Z39_50,cat__service_aol,cat__service_auth,cat__service_bgp,cat__service_courier,cat__service_csnet_ns,cat__service_ctf,cat__service_daytime,cat__service_discard,cat__service_domain,cat__service_domain_u,cat__service_echo,cat__service_eco_i,cat__service_ecr_i,cat__service_efs,cat__service_exec,cat__service_finger,cat__service_ftp,cat__service_ftp_data,cat__service_gopher,cat__service_harvest,cat__service_hostnames,cat__service_http,cat__service_http_2784,cat__service_http_443,cat__service_http_8001,cat__service_imap4,cat__service_iso_tsap,cat__service_klogin,cat__service_kshell,cat__service_ldap,cat__service_link,cat__service_login,cat__service_mtp,cat__service_name,cat__service_netbios_dgm,cat__service_netbios_ns,cat__service_netbios_ssn,cat__service_netstat,cat__service_nnsp,cat__service_nntp,cat__service_ntp_u,cat__service_other,cat__service_pm_dump,cat__service_pop_2,cat__service_pop_3,cat__service_printer,cat__service_private,cat__service_red_i,cat__service_remote_job,cat__service_rje,cat__service_shell,cat__service_smtp,cat__service_sql_net,cat__service_ssh,cat__service_sunrpc,cat__service_supdup,cat__service_systat,cat__service_telnet,cat__service_tftp_u,cat__service_tim_i,cat__service_time,cat__service_urh_i,cat__service_urp_i,cat__service_uucp,cat__service_uucp_path,cat__service_vmnet,cat__service_whois,cat__flag_OTH,cat__flag_REJ,cat__flag_RSTO,cat__flag_RSTOS0,cat__flag_RSTR,cat__flag_S0,cat__flag_S1,cat__flag_S2,cat__flag_S3,cat__flag_SF,cat__flag_SH
0,-0.110249,-0.007679,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,-0.027023,-0.809262,-0.011664,-0.036652,-0.024437,-0.012385,-0.02618,-0.01861,-0.041221,0.0,-0.002817,-0.097531,-0.717045,-0.354343,-0.637209,-0.631929,-0.374362,-0.374432,0.771283,-0.349683,-0.374560,-0.324063,-0.818890,-0.782367,-0.280282,0.069972,-0.289103,-0.639532,-0.624871,-0.224532,-0.376387,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.110249,-0.007737,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,-0.027023,-0.809262,-0.011664,-0.036652,-0.024437,-0.012385,-0.02618,-0.01861,-0.041221,0.0,-0.002817,-0.097531,-0.620982,-0.368110,-0.637209,-0.631929,-0.374362,-0.374432,-1.321428,0.482201,-0.374560,0.734343,-1.035688,-1.161030,2.736852,2.367737,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.110249,-0.007762,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,-0.027023,-0.809262,-0.011664,-0.036652,-0.024437,-0.012385,-0.02618,-0.01861,-0.041221,0.0,-0.002817,-0.097531,0.339648,-0.299273,1.602664,1.

## One-hot encoding: protocol_type

In [25]:
pd.concat([Xtr[["flag"]].head(5), clean_df.filter(like="cat__flag").head(5).astype(int)], axis=1)

,flag,cat__flag_OTH,cat__flag_REJ,cat__flag_RSTO,cat__flag_RSTOS0,cat__flag_RSTR,cat__flag_S0,cat__flag_S1,cat__flag_S2,cat__flag_S3,cat__flag_SF,cat__flag_SH
0,SF,0,0,0,0,0,0,0,0,0,1,0
1,SF,0,0,0,0,0,0,0,0,0,1,0
2,S0,0,0,0,0,0,1,0,0,0,0,0
3,SF,0,0,0,0,0,0,0,0,0,1,0
4,SF,0,0,0,0,0,0,0,0,0,1,0


## One-hot encoding: service (only the columns that are ticked in these 5 rows)

In [26]:
pd.concat([Xtr[["protocol_type"]].head(5), clean_df.filter(like="cat__protocol_type").head(5).astype(int)], axis=1)

,protocol_type,cat__protocol_type_icmp,cat__protocol_type_tcp,cat__protocol_type_udp
0,tcp,0,1,0
1,udp,0,0,1
2,tcp,0,1,0
3,tcp,0,1,0
4,tcp,0,1,0


## One-hot encoding: flag

In [27]:
t = clean_df.filter(like="cat__service").head(5).astype(int)
pd.concat([Xtr[["service"]].head(5), t.loc[:, t.any()]], axis=1)

,service,cat__service_ftp_data,cat__service_http,cat__service_other,cat__service_private
0,ftp_data,1,0,0,0
1,other,0,0,1,0
2,private,0,0,0,1
3,http,0,1,0,0
4,http,0,1,0,0


## Scaling: before and after

In [28]:
Xtr[["duration", "src_bytes", "dst_bytes", "count"]].head(5)

,duration,src_bytes,dst_bytes,count
0,0,491,0,2
1,0,146,0,13
2,0,0,0,123
3,0,232,8153,5
4,0,199,420,30


In [29]:
clean_df[["num__duration", "num__src_bytes", "num__dst_bytes", "num__count"]].head(5).round(3)

,num__duration,num__src_bytes,num__dst_bytes,num__count
0,-0.11,-0.008,-0.005,-0.717
1,-0.11,-0.008,-0.005,-0.621
2,-0.11,-0.008,-0.005,0.340
3,-0.11,-0.008,-0.003,-0.691
4,-0.11,-0.008,-0.005,-0.473


In [30]:
clean_df[["num__duration", "num__src_bytes", "num__dst_bytes", "num__count"]].agg(["mean", "std"]).round(3)

,num__duration,num__src_bytes,num__dst_bytes,num__count
mean,0.0,-0.0,0.0,-0.0
std,1.0,1.0,1.0,1.0


# 9. Define the models

In [31]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(max_depth=20, random_state=SEED),
    "Random Forest": RandomForestClassifier(150, n_jobs=-1, random_state=SEED),
    "Neural Net (MLP)": MLPClassifier((64, 32), early_stopping=True, max_iter=200, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, subsample=0.8,
                             colsample_bytree=0.8, eval_metric="logloss", n_jobs=-1, random_state=SEED),
}
list(models)

['Logistic Regression',
 'Decision Tree',
 'Random Forest',
 'Neural Net (MLP)',
 'XGBoost']

# 10. Train and test every model (binary: normal vs attack)

In [40]:
rows, fitted = [], {}
for name, m in models.items():
    pipe = make_pipeline(prep(), m)
    cv = cross_val_score(pipe, Xtr, tr.attack, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="f1")
    t = time.time(); pipe.fit(Xtr, tr.attack); fit = time.time() - t
    p = pipe.predict(Xte); s = pipe.predict_proba(Xte)[:, 1]
    tn, fp, fn, tp = confusion_matrix(te.attack, p).ravel()
    row = {"model": name, "cv_f1": cv.mean(), "cv_f1_std": cv.std(), "accuracy": accuracy_score(te.attack, p),
           "precision": precision_score(te.attack, p), "recall": recall_score(te.attack, p),
           "f1": f1_score(te.attack, p), "fpr": fp / (fp + tn), "roc_auc": roc_auc_score(te.attack, s),
           "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn), "train_s": fit}
    rows.append(row); fitted[name] = pipe
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()}, flush=True)
R["binary_models"] = rows

{'model': 'Logistic Regression', 'cv_f1': np.float64(0.9707), 'cv_f1_std': np.float64(0.0017), 'accuracy': 0.7539, 'precision': 0.9174, 'recall': 0.6239, 'f1': 0.7427, 'fpr': np.float64(0.0742), 'roc_auc': 0.7922, 'tp': 8006, 'fp': 721, 'tn': 8990, 'fn': 4827, 'train_s': 13.8995}
{'model': 'Decision Tree', 'cv_f1': np.float64(0.9983), 'cv_f1_std': np.float64(0.0003), 'accuracy': 0.7915, 'precision': 0.9656, 'recall': 0.6571, 'f1': 0.7821, 'fpr': np.float64(0.0309), 'roc_auc': 0.8133, 'tp': 8433, 'fp': 300, 'tn': 9411, 'fn': 4400, 'train_s': 15.1472}
{'model': 'Random Forest', 'cv_f1': np.float64(0.9988), 'cv_f1_std': np.float64(0.0002), 'accuracy': 0.7659, 'precision': 0.9671, 'recall': 0.6095, 'f1': 0.7478, 'fpr': np.float64(0.0274), 'roc_auc': 0.9603, 'tp': 7822, 'fp': 266, 'tn': 9445, 'fn': 5011, 'train_s': 25.4574}
{'model': 'Neural Net (MLP)', 'cv_f1': np.float64(0.9961), 'cv_f1_std': np.float64(0.0007), 'accuracy': 0.8086, 'precision': 0.9611, 'recall': 0.6917, 'f1': 0.8044, 'fpr

In [41]:
pd.DataFrame(rows)[["model", "cv_f1", "accuracy", "precision", "recall", "f1", "fpr", "roc_auc", "train_s"]].round(4)

,model,cv_f1,accuracy,precision,recall,f1,fpr,roc_auc,train_s
0,Logistic Regression,0.9707,0.7539,0.9174,0.6239,0.7427,0.0742,0.7922,13.8995
1,Decision Tree,0.9983,0.7915,0.9656,0.6571,0.7821,0.0309,0.8133,15.1472
2,Random Forest,0.9988,0.7659,0.9671,0.6095,0.7478,0.0274,0.9603,25.4574
3,Neural Net (MLP),0.9961,0.8086,0.9611,0.6917,0.8044,0.0370,0.9085,162.1156
4,XGBoost,0.9991,0.7901,0.9686,0.6523,0.7796,0.0279,0.9673,13.0817


## Look at one model's predictions (XGBoost)

In [42]:
pipe = fitted["XGBoost"]
p = pipe.predict(Xte); s = pipe.predict_proba(Xte)[:, 1]
res = pd.DataFrame({"label": te.label, "truth": te.attack, "guess": p, "attack_prob": s.round(3)})
res.head(10)

,label,truth,guess,attack_prob
0,neptune,1,1,1.000
1,neptune,1,1,1.000
2,normal,0,0,0.000
3,saint,1,1,1.000
4,mscan,1,0,0.404
5,normal,0,0,0.000
6,normal,0,0,0.002
7,guess_passwd,1,0,0.001
8,normal,0,0,0.000
9,guess_passwd,1,0,0.000


In [43]:
res[res.truth != res.guess].head(10)

,label,truth,guess,attack_prob
4,mscan,1,0,0.404
7,guess_passwd,1,0,0.001
9,guess_passwd,1,0,0.000
10,mscan,1,0,0.002
33,normal,0,1,1.000
35,mscan,1,0,0.035
37,buffer_overflow,1,0,0.016
54,warezmaster,1,0,0.040
57,snmpgetattack,1,0,0.001
58,processtable,1,0,0.004


In [44]:
tn, fp, fn, tp = confusion_matrix(te.attack, p).ravel()
pd.DataFrame([[tn, fp], [fn, tp]], index=["really normal", "really attack"], columns=["says normal", "says attack"])

,says normal,says attack
really normal,9440,271
really attack,4462,8371


In [45]:
res["family"] = te.family
res.assign(correct=(res.truth == res.guess)).groupby("family").correct.mean().round(3)

family
DoS       0.866
Normal    0.972
Probe     0.715
R2L       0.060
U2R       0.149
Name: correct, dtype: float64

# 11. Five-class model (Normal / DoS / Probe / R2L / U2R)

In [46]:
best = max(rows, key=lambda r: r["cv_f1"])["model"]
mc = make_pipeline(prep(), type(models[best])(**models[best].get_params()))
labels = sorted(tr.family.unique()); lab = {l: i for i, l in enumerate(labels)}
best, lab

('XGBoost', {'DoS': 0, 'Normal': 1, 'Probe': 2, 'R2L': 3, 'U2R': 4})

In [47]:
mc.fit(Xtr, tr.family.map(lab)); pm = mc.predict(Xte)
inv = {i: l for l, i in lab.items()}
pd.DataFrame({"label": te.label, "true_family": te.family, "guess": [inv[i] for i in pm]}).head(10)

,label,true_family,guess
0,neptune,DoS,DoS
1,neptune,DoS,DoS
2,normal,Normal,Normal
3,saint,Probe,Probe
4,mscan,Probe,Normal
5,normal,Normal,Normal
6,normal,Normal,Normal
7,guess_passwd,R2L,Normal
8,normal,Normal,Normal
9,guess_passwd,R2L,Normal


In [48]:
cm5 = confusion_matrix(te.family.map(lab), pm)
pd.DataFrame(cm5, index=["really " + l for l in labels], columns=["says " + l for l in labels])

,says DoS,says Normal,says Probe,says R2L,says U2R
really DoS,6276,1068,116,0,0
really Normal,67,9442,200,1,1
really Probe,165,696,1560,0,0
really R2L,0,2723,3,157,2
really U2R,0,58,0,0,9


In [49]:
rep = classification_report(te.family.map(lab), pm, target_names=labels, output_dict=True, zero_division=0)
pd.DataFrame(rep).T.round(3)

,precision,recall,f1-score,support
DoS,0.964,0.841,0.899,7460.000
Normal,0.675,0.972,0.797,9711.000
Probe,0.830,0.644,0.726,2421.000
R2L,0.994,0.054,0.103,2885.000
U2R,0.750,0.134,0.228,67.000
accuracy,0.774,0.774,0.774,0.774
macro avg,0.843,0.529,0.550,22544.000
weighted avg,0.828,0.774,0.732,22544.000


# 12. Save results

In [50]:
R["multiclass"] = {"model": best, "accuracy": accuracy_score(te.family.map(lab), pm),
                   "macro_f1": f1_score(te.family.map(lab), pm, average="macro"),
                   "per_class": {k: {m: round(v, 4) for m, v in rep[k].items()} for k in labels}}
R["selected_by_cv_f1"] = best
print(json.dumps(R["multiclass"], indent=1))
json.dump(R, open("results.json", "w"), indent=1, default=float)
print("\nDONE. Saved results.json")

{
 "model": "XGBoost",
 "accuracy": 0.7737757274662882,
 "macro_f1": 0.5504206128994039,
 "per_class": {
  "DoS": {
   "precision": 0.9644,
   "recall": 0.8413,
   "f1-score": 0.8986,
   "support": 7460.0
  },
  "Normal": {
   "precision": 0.6751,
   "recall": 0.9723,
   "f1-score": 0.7969,
   "support": 9711.0
  },
  "Probe": {
   "precision": 0.8302,
   "recall": 0.6444,
   "f1-score": 0.7256,
   "support": 2421.0
  },
  "R2L": {
   "precision": 0.9937,
   "recall": 0.0544,
   "f1-score": 0.1032,
   "support": 2885.0
  },
  "U2R": {
   "precision": 0.75,
   "recall": 0.1343,
   "f1-score": 0.2278,
   "support": 67.0
  }
 }
}

DONE. Saved results.json
